# Week 3, Classification: Counting with Weights

**What you'll do today.** You'll train a transparent classifier, a logistic regression, on a
pre-labeled pop corpus, then **read its mind**: the signed weights that are its most positive
and most negative words.

> **Don't lose your work.** Opened from GitHub, this notebook is read-only: **File → Save a copy in Drive** before editing, and save durable outputs to your Drive project folder; Colab's own disk is wiped when the runtime ends. Course notebooks get updates during the term; to pick them up, open the notebook fresh from GitHub (or `git pull` if you cloned the repo). Updates never touch your saved copy.


In [ ]:
# If an import fails: re-run the install cell above; if it persists see ../kits/common-errors-cheatsheet.md
# (standalone copy: https://github.com/lucianli123/culture-as-data-2026/blob/main/kits/common-errors-cheatsheet.md)
# --- Make your work survive a Colab reset -------------------------------------
# Colab wipes the runtime when it disconnects or idles out. Mount your Google Drive
# and keep everything in ONE project folder, so your corpus, embeddings, labels, and
# charts are still there next week. (Outside Colab - e.g. the offline test harness -
# this falls back to a local folder so the notebook still runs.)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

In [ ]:
# Fetch the pinned package lists if they aren't beside the notebook (this happens
# whenever you open a single notebook straight from GitHub in Colab).
import os, urllib.request

def _bad(p):
    """Missing, empty, or corrupted (a crashed runtime can leave NUL-padded files)."""
    if not os.path.exists(p): return True
    data = open(p, "rb").read()
    return len(data) == 0 or b"\x00" in data

_RAW = "https://raw.githubusercontent.com/lucianli123/culture-as-data-2026/main/notebooks/"
for _f in ("requirements.txt", "constraints.txt"):
    if _bad(_f):
        urllib.request.urlretrieve(_RAW + _f, _f)
assert not _bad("requirements.txt"), "download failed; re-run this cell"

# First, install the few packages Colab doesn't already ship. If you opened this
# notebook with the whole repo, the line below installs them:
%pip install -q -r requirements.txt

# Opened this notebook on its own, without the repo files? Comment the line above
# and use this explicit pinned install instead:
# %pip install -q pandas scikit-learn matplotlib
# If an import later fails with "numpy.dtype size changed" or similar: the install
# replaced a compiled library under an already-loaded module. Runtime -> Restart
# session, then run again from the top. It should not recur.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
print("imports ok")

In [ ]:
import os

## A real labeled corpus: one city, two subreddits

r/sandiego and r/SanDiegan are two communities discussing the same city (the second
split off from the first). Same beaches, same rents, largely the same words, which makes
this a hard, honest test: **can a classifier tell them apart, and which words give a
comment away?** With different-topic pairs the answer is easy and boring; here the
accuracy is modest and the weights are the finding.

Loaded live (nothing is stored in the repo; community text is analyze-only). If the
archive API is unavailable, the notebook falls back to two big archived subreddits, then
to two novelists from Gutenberg. The mechanics are identical either way.


In [ ]:
import re

SUB_A, SUB_B = "sandiego", "SanDiegan"   # one city, two communities

def load_arctic(a, b, n=500):
    """Any two subreddits, via the Arctic Shift archive API (analyze-only).
    Retries politely: a whole classroom at once gets rate-limited."""
    import requests, time
    def pull(sub, tries=3):
        for attempt in range(tries):
            got, before, pages = [], None, 0
            try:
                while len(got) < n and pages < 12:   # ~15 seconds per subreddit at n=500
                    params = {"subreddit": sub, "limit": 100, "fields": "body,created_utc"}
                    if before: params["before"] = before
                    resp = requests.get("https://arctic-shift.photon-reddit.com/api/comments/search",
                                        params=params, timeout=30)
                    resp.raise_for_status()
                    rows = resp.json()["data"]
                    pages += 1
                    if not rows: break
                    before = int(min(r["created_utc"] for r in rows))
                    got += [r["body"] for r in rows if isinstance(r.get("body"), str)
                            and 80 < len(r["body"]) < 800
                            and "[removed]" not in r["body"] and "[deleted]" not in r["body"]]
                if got:
                    return got[:n]
            except Exception as e:
                print(f"r/{sub}: {type(e).__name__} (attempt {attempt + 1}/{tries}), waiting...")
            time.sleep(5 * (attempt + 1))
        return []
    a_txt, b_txt = pull(a), pull(b)
    if not (a_txt and b_txt): raise ValueError("empty pull")
    return pd.DataFrame({"text": a_txt + b_txt, "label": [a] * len(a_txt) + [b] * len(b_txt)})

def load_subreddits(a, b, n=200):
    """Alternative: 50 big subreddits archived on a public mirror (streaming, analyze-only)."""
    from itertools import islice
    from datasets import load_dataset
    texts, labels = [], []
    for sub in (a, b):
        ds = load_dataset("parquet", split="train", streaming=True,
            data_files=f"hf://datasets/HuggingFaceGECLM/REDDIT_comments/data/{sub}-*.parquet")
        stream = (r["body"] for r in ds)
        good = (t for t in stream if isinstance(t, str) and 30 < len(t) < 300
                and "[removed]" not in t and "[deleted]" not in t)
        got = list(islice(good, n))
        texts += got; labels += [sub] * len(got)
    return pd.DataFrame({"text": texts, "label": labels})

def load_gutenberg(url):
    """Fetch from Project Gutenberg and strip the boilerplate."""
    import requests
    raw = requests.get(url, timeout=30).text.replace("\r\n", "\n")
    body = re.split(r"\*\*\* ?START OF (?:THE|THIS) PROJECT GUTENBERG.*?\*\*\*", raw, flags=re.S)[-1]
    return re.split(r"\*\*\* ?END OF (?:THE|THIS) PROJECT GUTENBERG", body)[0]

def load_novelists(n=400):
    """The last-resort pair: Shelley against Stoker, fetched live from Gutenberg."""
    def sents(url):
        t = load_gutenberg(url)
        return [x.strip().replace("\n", " ") for x in re.split(r"(?<=[.!?])\s+", t) if 40 < len(x) < 180][:n]
    a = sents("https://www.gutenberg.org/cache/epub/84/pg84.txt")
    b = sents("https://www.gutenberg.org/cache/epub/345/pg345.txt")
    return pd.DataFrame({"text": a + b, "label": ["shelley"] * len(a) + ["stoker"] * len(b)})

try:
    df = load_arctic(SUB_A, SUB_B)
    LABEL_A, LABEL_B = SUB_A, SUB_B
    print(f"live corpus: r/{SUB_A} vs r/{SUB_B}")
except Exception as e:
    print("archive API unavailable:", type(e).__name__)
    try:
        df = load_subreddits("askscience", "gaming")
        LABEL_A, LABEL_B = "askscience", "gaming"
        print("using the mirror pair instead: r/askscience vs r/gaming")
    except Exception as e2:
        print("mirror unavailable too, novelists it is:", type(e2).__name__)
        df = load_novelists()
        LABEL_A, LABEL_B = "shelley", "stoker"
print(df["label"].value_counts().to_string())
df.sample(4, random_state=1)


## Train it, the AI writes this; reading the result is your job

Each word becomes a column, the model learns a weight (a vote, for or against), and it adds them
up.

In [ ]:
vec = CountVectorizer()
X = vec.fit_transform(df["text"])
y = (df["label"] == LABEL_A).astype(int)

# Hold out a quarter of the rows the model never sees; accuracy is measured there:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print("held-out accuracy:", round(accuracy_score(yte, clf.predict(Xte)), 3))
clf_full = LogisticRegression(max_iter=1000).fit(X, y)  # refit on all for weight-reading

## Read its mind, the signed weights

The most **positive** and most **negative** words are the model's mind on the table.

In [ ]:
terms = np.array(vec.get_feature_names_out())
w = clf_full.coef_.ravel()
order = w.argsort()
print("Most NEGATIVE words:", ", ".join(terms[order[:6]]))
print("Most POSITIVE words:", ", ".join(terms[order[-6:][::-1]]))

plt.figure(figsize=(6,3))
top = np.concatenate([order[:6], order[-6:]])
plt.barh(terms[top], w[top])
plt.title("The classifier's signed weights")
plt.tight_layout(); plt.show()

### Build it a "black cat"

The Teachable Machine demo learned a bias from a skewed training set (orange cats, brown dogs).

In [ ]:
def predict(s):
    p = clf_full.predict_proba(vec.transform([s]))[0, 1]
    return f"{LABEL_A if p > 0.5 else LABEL_B} (p_{LABEL_A}={p:.2f})"

for label in (LABEL_A, LABEL_B):
    example = df[df["label"] == label]["text"].iloc[-1]
    print(f"held-out-ish {label!r:14}:", predict(example))
print("the black cat (Shakespeare - neither class):",
      predict("shall i compare thee to a summer's day"))
# The model has two boxes and no concept of "neither." Every classifier you will ever
# meet has this property; the question is only how confidently it hides it.

### The words that give each side away

Before the caveats: the stylistic fingerprint the machine learned, on the table.

## Playground: compare your own two subreddits

The whole pipeline, one call. Pick any two communities you actually know; the archive
covers essentially all of Reddit. Pairs that teach: two communities you'd expect to be
identical, a hobby and its neighbor (r/coffee vs r/espresso), two cities, two fandoms.
Write your question first, then read the weights, not just the accuracy.


In [ ]:
# MY QUESTION: ...one sentence here...

def compare_subreddits(a, b, n=400):
    """Load two subreddits, train the same classifier as above, report what it learned."""
    d = load_arctic(a, b, n=n)
    v = CountVectorizer(min_df=3)
    Xc = v.fit_transform(d["text"]); yc = (d["label"] == a).astype(int)
    Xtr_, Xte_, ytr_, yte_ = train_test_split(Xc, yc, test_size=0.25, random_state=0, stratify=yc)
    m = LogisticRegression(max_iter=2000).fit(Xtr_, ytr_)
    print(f"r/{a} vs r/{b}: {d['label'].value_counts().to_dict()}")
    print(f"held-out accuracy: {m.score(Xte_, yte_):.2f}  (0.50 = coin flip)")
    t = np.array(v.get_feature_names_out()); w = m.coef_.ravel()
    print(f"most r/{a}:", ", ".join(t[w.argsort()[::-1][:10]]))
    print(f"most r/{b}:", ", ".join(t[w.argsort()[:10]]))
    return m, v

# The demo pair, then yours. Swap in any two subreddit names and rerun:
compare_subreddits("soccer", "nba", n=300)
# compare_subreddits("coffee", "espresso")
# compare_subreddits("AskMen", "AskWomen")
# compare_subreddits("your_pick_a", "your_pick_b")

# Reading list for the result: Is the accuracy near 0.5 (the communities talk alike)
# or near 1.0 (different worlds)? Are the top words topic (nba: lebron), register
# (askscience: citation), or community habits (this sub, mods)? That reading IS the
# finding; the accuracy alone is not.
